In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Add project source directory to import path.
import sys
import os
from pathlib import Path

parent_dir = str(os.path.join(Path.cwd().parent, 'src'))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Imports.
from typing import cast
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import fvopn_simulations as fvopn

In [ ]:
rng = np.random.default_rng(42)
alpha = 1 # Root slots.
k = 2 # Multiplication factor.
size = 1000 # Number of nodes.
budgets=rng.beta(.5, 8, size) # Random generated budgets.
cost_fn = lambda b, sel, m: min(b[sel, 2][0], b[:, 2][0].sum() * (1 / b[sel, 1][0]))
                    

conf = fvopn.VoteAuctionModelV2Config(
    budgets=budgets,
    root_slots=alpha,
    multiplication_factor=k,
    cost=cost_fn,
)

model = fvopn.VoteAuctionModelV2(conf, rng)

In [ ]:
def graph_fn(up_to: int, fn):
    fn_vec = np.vectorize(fn)

    xs = np.arange(1, up_to)
    ys = fn_vec(xs)

    return (xs, ys)

In [ ]:
def V(whale_budget: float, x: int):
    b = np.copy(budgets)

    b[x] = whale_budget

    b[0:x] = whale_budget / x
    
    conf = fvopn.VoteAuctionModelV2Config(
        budgets=b,
        root_slots=alpha,
        multiplication_factor=k,
    )

    model = fvopn.VoteAuctionModelV2(conf, rng)

    res = 0

    N = 1
    
    for _ in range(N):
        a = model.run()

        a = a.sort_index()
        
        res += a.loc[0:x, 'votes'].sum()

    res /= N

    return res

In [ ]:
up_to = 50

x1, y1 = graph_fn(up_to, lambda x: V(np.max(budgets[x+1:]), x))
x2, y2 = graph_fn(up_to, lambda x: V(np.max(budgets[x+1:])*2, x))
x3, y3 = graph_fn(up_to, lambda x: V(np.max(budgets[x+1:])*3, x))
x4, y4 = graph_fn(up_to, lambda x: V(np.max(budgets[x+1:])*4, x))

In [ ]:
plt.plot(x1, y1, label="max")
plt.plot(x2, y2, label="2*max")
plt.plot(x3, y3, label="3*max")
plt.plot(x4, y4, label="4*max")
plt.xticks(np.arange(0, up_to, 5))
plt.title("Resistencia a Fraude")
plt.xlabel("Identidades")
plt.ylabel("Influencia")
plt.grid()
plt.legend()
plt.show()